# Exercises XP Gold ? Prompt Engineering
Last Updated: October 7th, 2025
Use this guided notebook and fill each TODO before running cells.

## What you'll learn
- Craft precise prompts for complex tasks.
- Debug and refine chain-of-thought (CoT) reasoning.
- Select prompt patterns for real-world applications and reduce hallucinations.
- Design chained prompts with conditional logic, bias mitigation, and simulated memory.

## What you'll build
- Corrected CoT prompts and answers.
- Domain-specific prompt patterns and multi-step pipelines.
- Fair role-based prompts and conversational agents with memory.

## Helper: Run a prompt
Uses `ollama run` if available (set `OLLAMA_MODEL` to override, default: `llama3`). If Ollama is missing or the call fails, the helper prints the prompt in dry-run mode instead of crashing.


In [1]:
import os, subprocess

def run_prompt(prompt: str, model: str | None = None, temperature: float = 0.7, max_new_tokens: int = 120):
    """Send a single-turn prompt via `ollama run`.

    - Set OLLAMA_MODEL env var or pass model to override.
    - Falls back to dry-run if ollama is unavailable."""
    model = model or os.environ.get('OLLAMA_MODEL', 'llama3')
    cmd = ['ollama', 'run', model]
    try:
        proc = subprocess.run(cmd, input=prompt.encode('utf-8'), stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=True)
        out = proc.stdout.decode('utf-8', errors='ignore')
        print(out)
        return out
    except FileNotFoundError:
        print('[dry-run] ollama not installed. Prompt to send:', prompt)
    except subprocess.CalledProcessError as e:
        err = e.stderr.decode('utf-8', errors='ignore') if e.stderr else str(e)
        print('[dry-run] ollama call failed:', err)
    return None


## Exercise 1: Debug a Faulty Chain-of-Thought
Goal: Fix the CoT and final answer. Original prompt had math/logic issues for the pencil change problem.

### Tasks
- Find the reasoning mistake.
- Rewrite the CoT prompt with correct steps.
- Provide the correct answer.

In [2]:
# TODO: describe the mistake
cot_issue = '''
L'erreur se trouve dans le calcul du coût total.
Le raisonnement indique que 6 × 0,75 $ = 4,75 $, alors que le résultat correct est 4,50 $.
Par conséquent, le calcul de la monnaie est également incorrect.
'''
cot_issue

"\nL'erreur se trouve dans le calcul du coût total.\nLe raisonnement indique que 6 × 0,75 $ = 4,75 $, alors que le résultat correct est 4,50 $.\nPar conséquent, le calcul de la monnaie est également incorrect.\n"

In [3]:
# TODO: rewrite the CoT prompt
fixed_cot_prompt = '''
Résolvons le problème étape par étape.

1. Calculer le coût total des crayons :
   6 × 0,75 $ = 4,50 $

2. Calculer la monnaie rendue :
   5,00 $ - 4,50 $ = 0,50 $

Conclusion :
Alice reçoit 0,50 $ de monnaie.
'''
fixed_cot_prompt

'\nRésolvons le problème étape par étape.\n\n1. Calculer le coût total des crayons :\n   6 × 0,75 $ = 4,50 $\n\n2. Calculer la monnaie rendue :\n   5,00 $ - 4,50 $ = 0,50 $\n\nConclusion :\nAlice reçoit 0,50 $ de monnaie.\n'

In [4]:
import os, subprocess

def run_prompt(prompt: str, model: str | None = None, temperature: float = 0.7, max_new_tokens: int = 120):
    """Send a single-turn prompt via `ollama run`.

    - Set OLLAMA_MODEL env var or pass model to override.
    - Falls back to dry-run if ollama is unavailable."""
    model = model or os.environ.get('OLLAMA_MODEL', 'llama3')
    cmd = ['ollama', 'run', model]
    try:
        proc = subprocess.run(cmd, input=prompt.encode('utf-8'), stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=True)
        out = proc.stdout.decode('utf-8', errors='ignore')
        print(out)
        return out
    except FileNotFoundError:
        print('[dry-run] ollama not installed. Prompt to send:', prompt)
    except subprocess.CalledProcessError as e:
        err = e.stderr.decode('utf-8', errors='ignore') if e.stderr else str(e)
        print('[dry-run] ollama call failed:', err)
    return None


In [5]:
# TODO: final correct answer
correct_change = '0,50 $'
correct_change

'0,50 $'

## Exercise 2: Choose the Right Prompt Pattern
Goal: Pick and justify a prompt pattern for support ticket classification (Billing Issue, Technical Support, Account Access, Other).

### Tasks
- Choose a pattern (Zero-Shot, Few-Shot, IAP, LoT, etc.).
- Write a complete prompt using that pattern.
- Justify the choice (ambiguity, consistency, generalization).

In [6]:
# TODO: choose pattern and prompt
chosen_pattern = 'Few-Shot Prompting'
classification_prompt = '''
Vous êtes un assistant chargé de classer les messages des clients dans une seule des catégories suivantes :
- Problème de facturation
- Assistance technique
- Accès au compte
- Autre

Exemples :

Client : "J'ai été débité deux fois pour mon abonnement."
Catégorie : Problème de facturation

Client : "Mon application se ferme dès que je l'ouvre."
Catégorie : Assistance technique

Client : "J'ai oublié mon mot de passe et je ne peux plus me connecter."
Catégorie : Accès au compte

Client : "Quels sont vos horaires d'ouverture ?"
Catégorie : Autre

Consigne :
Pour le message suivant, réponds uniquement par le nom de la catégorie.

Client : "{message_client}"
Catégorie :
'''
justification = '''
Le Few-Shot Prompting est le choix le plus approprié car il fournit au modèle
plusieurs exemples représentatifs de chaque catégorie. Cela réduit les ambiguïtés,
améliore la cohérence des classifications et permet au modèle de mieux généraliser
à de nouveaux messages, même lorsque leur formulation diffère des exemples.
Contrairement au Zero-Shot, il guide explicitement le modèle sur les attentes de
classification sans nécessiter d'entraînement supplémentaire.
'''
classification_prompt

'\nVous êtes un assistant chargé de classer les messages des clients dans une seule des catégories suivantes :\n- Problème de facturation\n- Assistance technique\n- Accès au compte\n- Autre\n\nExemples :\n\nClient : "J\'ai été débité deux fois pour mon abonnement."\nCatégorie : Problème de facturation\n\nClient : "Mon application se ferme dès que je l\'ouvre."\nCatégorie : Assistance technique\n\nClient : "J\'ai oublié mon mot de passe et je ne peux plus me connecter."\nCatégorie : Accès au compte\n\nClient : "Quels sont vos horaires d\'ouverture ?"\nCatégorie : Autre\n\nConsigne :\nPour le message suivant, réponds uniquement par le nom de la catégorie.\n\nClient : "{message_client}"\nCatégorie :\n'

In [8]:
import os, subprocess

def run_prompt(prompt: str, model: str | None = None, temperature: float = 0.7, max_new_tokens: int = 120):
    """Send a single-turn prompt via `ollama run`.

    - Set OLLAMA_MODEL env var or pass model to override.
    - Falls back to dry-run if ollama is unavailable."""
    model = model or os.environ.get('OLLAMA_MODEL', 'llama3')
    cmd = ['ollama', 'run', model]
    try:
        proc = subprocess.run(cmd, input=prompt.encode('utf-8'), stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=True)
        out = proc.stdout.decode('utf-8', errors='ignore')
        print(out)
        return out
    except FileNotFoundError:
        print('[dry-run] ollama not installed. Prompt to send:', prompt)
    except subprocess.CalledProcessError as e:
        err = e.stderr.decode('utf-8', errors='ignore') if e.stderr else str(e)
        print('[dry-run] ollama call failed:', err)
    return None


## Exercise 3: Use AlignedCoT to Compare Reasoning Paths
Goal: Two reasoning paths + comparison for the flower pot cost.

### Tasks
- Build AlignedCoT with at least two distinct reasoning structures.
- Add a comparison step to select the consistent answer.

In [11]:
# TODO: aligned CoT prompt
aligned_cot_prompt = '''
Tu es un assistant chargé de résoudre un problème de manière fiable.

Problème :
Un jardinier achète :
- 2 petits pots à 2 $ chacun,
- 3 pots moyens à 4 $ chacun,
- 1 grand pot à 6 $.

Utilise la méthode Aligned Chain-of-Thought en suivant les étapes ci-dessous :

Chemin de raisonnement 1 :
- Calcule le coût de chaque catégorie de pots.
- Additionne ensuite les trois coûts pour obtenir le total.

Chemin de raisonnement 2 :
- Représente chaque achat sous forme d'une multiplication.
- Écris ensuite une seule expression :
  (2 × 2) + (3 × 4) + (1 × 6)
- Calcule le résultat.

Comparaison :
- Compare les résultats obtenus par les deux méthodes.
- Vérifie qu'ils sont identiques.
- En cas de différence, identifie l'erreur et conserve uniquement le raisonnement cohérent.

Réponse finale :
Indique uniquement le coût total en dollars.
'''
aligned_cot_prompt

"\nTu es un assistant chargé de résoudre un problème de manière fiable.\n\nProblème :\nUn jardinier achète :\n- 2 petits pots à 2 $ chacun,\n- 3 pots moyens à 4 $ chacun,\n- 1 grand pot à 6 $.\n\nUtilise la méthode Aligned Chain-of-Thought en suivant les étapes ci-dessous :\n\nChemin de raisonnement 1 :\n- Calcule le coût de chaque catégorie de pots.\n- Additionne ensuite les trois coûts pour obtenir le total.\n\nChemin de raisonnement 2 :\n- Représente chaque achat sous forme d'une multiplication.\n- Écris ensuite une seule expression :\n  (2 × 2) + (3 × 4) + (1 × 6)\n- Calcule le résultat.\n\nComparaison :\n- Compare les résultats obtenus par les deux méthodes.\n- Vérifie qu'ils sont identiques.\n- En cas de différence, identifie l'erreur et conserve uniquement le raisonnement cohérent.\n\nRéponse finale :\nIndique uniquement le coût total en dollars.\n"

In [12]:
import os, subprocess

def run_prompt(prompt: str, model: str | None = None, temperature: float = 0.7, max_new_tokens: int = 120):
    """Send a single-turn prompt via `ollama run`.

    - Set OLLAMA_MODEL env var or pass model to override.
    - Falls back to dry-run if ollama is unavailable."""
    model = model or os.environ.get('OLLAMA_MODEL', 'llama3')
    cmd = ['ollama', 'run', model]
    try:
        proc = subprocess.run(cmd, input=prompt.encode('utf-8'), stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=True)
        out = proc.stdout.decode('utf-8', errors='ignore')
        print(out)
        return out
    except FileNotFoundError:
        print('[dry-run] ollama not installed. Prompt to send:', prompt)
    except subprocess.CalledProcessError as e:
        err = e.stderr.decode('utf-8', errors='ignore') if e.stderr else str(e)
        print('[dry-run] ollama call failed:', err)
    return None


## Exercise 4: Design a Multi-Step Document Pipeline
Goal: Three-stage pipeline (domain detect -> contributions -> follow-up question) with conditional/context chaining.

### Tasks
- Draft prompt template for each stage.
- Note where to chain context or branch conditionally.

In [13]:
# TODO: stage prompts
stage1_domain = '''
Vous êtes un expert en classification d'articles scientifiques.

Lisez le résumé ci-dessous et identifiez le domaine principal de recherche.

Choisissez uniquement une catégorie parmi :
- Biologie
- Physique
- Informatique
- Mathématiques
- Chimie
- Autre

Résumé :
{abstract}

Répondez uniquement par le nom du domaine.
'''
stage2_contrib = '''
Vous êtes un assistant de recherche.

À partir du résumé suivant, extrayez les principales contributions de l'article.

Résumé :
{abstract}

Consignes :
- Identifiez 3 contributions au maximum.
- Utilisez des phrases courtes.
- Ne faites aucune supposition au-delà du résumé.
'''
stage3_followup = '''Vous êtes un chercheur.

En vous basant sur :
- le domaine identifié : {domain}
- les principales contributions : {contributions}

Proposez une question de recherche pertinente qui pourrait prolonger les travaux présentés.

La question doit :
- être claire,
- être réaliste,
- être liée aux contributions de l'article.'''
chaining_notes = '''
Pipeline de traitement :

1. Étape 1 : Identifier automatiquement le domaine de recherche à partir du résumé.
2. Étape 2 : Utiliser le même résumé pour extraire les principales contributions.
3. Étape 3 : Réutiliser le domaine identifié et les contributions extraites afin de générer une nouvelle question de recherche pertinente.

Cette approche par Prompt Chaining rend le processus modulaire, améliore la qualité des résultats et facilite le remplacement ou l'amélioration d'une étape sans modifier les autres.
'''
stage1_domain, stage2_contrib, stage3_followup

("\nVous êtes un expert en classification d'articles scientifiques.\n\nLisez le résumé ci-dessous et identifiez le domaine principal de recherche.\n\nChoisissez uniquement une catégorie parmi :\n- Biologie\n- Physique\n- Informatique\n- Mathématiques\n- Chimie\n- Autre\n\nRésumé :\n{abstract}\n\nRépondez uniquement par le nom du domaine.\n",
 "\nVous êtes un assistant de recherche.\n\nÀ partir du résumé suivant, extrayez les principales contributions de l'article.\n\nRésumé :\n{abstract}\n\nConsignes :\n- Identifiez 3 contributions au maximum.\n- Utilisez des phrases courtes.\n- Ne faites aucune supposition au-delà du résumé.\n",
 "Vous êtes un chercheur.\n\nEn vous basant sur :\n- le domaine identifié : {domain}\n- les principales contributions : {contributions}\n\nProposez une question de recherche pertinente qui pourrait prolonger les travaux présentés.\n\nLa question doit :\n- être claire,\n- être réaliste,\n- être liée aux contributions de l'article.")

## Exercise 5: Role Prompting to Reduce Bias
Goal: Fair career recommendations.

### Tasks
- Write a basic prompt (may be biased).
- Write a revised role-based prompt to reduce bias.
- Explain how the role prompt improves fairness.

In [14]:
# TODO: basic and revised prompts
biased_prompt = 'Recommande un parcours professionnel à un utilisateur en fonction de ses compétences et de ses centres d\'intérêt.'
debiased_prompt = '''
Tu es un conseiller d'orientation professionnelle impartial et inclusif.

Ta mission est de recommander des parcours professionnels uniquement à partir
des compétences, des intérêts, des expériences et des objectifs de l'utilisateur.

Consignes :
- N'utilise jamais le genre, l'âge, l'origine, la nationalité ou tout autre
  stéréotype pour influencer tes recommandations.
- Base-toi uniquement sur les informations fournies.
- Si plusieurs parcours sont pertinents, présente plusieurs options avec leurs avantages.
- Explique brièvement pourquoi chaque parcours correspond au profil de l'utilisateur.
- Si certaines informations sont insuffisantes, demande des précisions avant de faire une recommandation.
'''
fairness_explanation = '''
La première consigne est très générale et laisse le modèle interpréter librement
les critères de recommandation. Il peut donc reproduire des biais présents dans
ses données d'entraînement, par exemple associer certains métiers à un genre ou
à un groupe particulier.

La seconde consigne utilise une incitation par rôle en demandant au modèle
d'agir comme un conseiller d'orientation impartial et inclusif. Elle précise
explicitement que les recommandations doivent être fondées uniquement sur les
compétences, les centres d'intérêt et les objectifs de l'utilisateur, tout en
interdisant l'utilisation de stéréotypes. Cette approche améliore l'équité, la
cohérence et la transparence des recommandations.
'''
debiased_prompt

"\nTu es un conseiller d'orientation professionnelle impartial et inclusif.\n\nTa mission est de recommander des parcours professionnels uniquement à partir\ndes compétences, des intérêts, des expériences et des objectifs de l'utilisateur.\n\nConsignes :\n- N'utilise jamais le genre, l'âge, l'origine, la nationalité ou tout autre\n  stéréotype pour influencer tes recommandations.\n- Base-toi uniquement sur les informations fournies.\n- Si plusieurs parcours sont pertinents, présente plusieurs options avec leurs avantages.\n- Explique brièvement pourquoi chaque parcours correspond au profil de l'utilisateur.\n- Si certaines informations sont insuffisantes, demande des précisions avant de faire une recommandation.\n"

In [15]:
import os, subprocess

def run_prompt(prompt: str, model: str | None = None, temperature: float = 0.7, max_new_tokens: int = 120):
    """Send a single-turn prompt via `ollama run`.

    - Set OLLAMA_MODEL env var or pass model to override.
    - Falls back to dry-run if ollama is unavailable."""
    model = model or os.environ.get('OLLAMA_MODEL', 'llama3')
    cmd = ['ollama', 'run', model]
    try:
        proc = subprocess.run(cmd, input=prompt.encode('utf-8'), stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=True)
        out = proc.stdout.decode('utf-8', errors='ignore')
        print(out)
        return out
    except FileNotFoundError:
        print('[dry-run] ollama not installed. Prompt to send:', prompt)
    except subprocess.CalledProcessError as e:
        err = e.stderr.decode('utf-8', errors='ignore') if e.stderr else str(e)
        print('[dry-run] ollama call failed:', err)
    return None


## Exercise 6: Build a Conversational Agent with Context Memory
Goal: Add memory to a virtual health coach.

### Tasks
- Pick a memory technique (prior message passing, structured history, vector store retrieval).
- Show how you would structure past context.
- Write a prompt that injects that context for the next response.

In [16]:
# TODO: choose memory approach and build context + prompt
memory_approach = 'Historique structuré (Structured Conversation History)'
past_context = '''
Contexte utilisateur :

Nom : Alex

Préférences :
- Souhaite améliorer la qualité de son sommeil.
- Préfère les exercices de faible intensité.
- Essaie de réduire sa consommation de sucre.

Résumé des interactions précédentes :
- L'utilisateur dort en moyenne 6 heures par nuit.
- Le coach a conseillé de viser 7 à 8 heures de sommeil.
- Le coach a recommandé d'éviter les écrans 30 minutes avant le coucher.
- Le coach a suggéré une marche de 30 minutes, 5 jours par semaine.
- L'utilisateur a accepté d'essayer ces recommandations.
'''
next_turn_prompt = '''
Tu es un coach de santé virtuel.

Utilise le contexte suivant pour personnaliser ta réponse.

Contexte :
{past_context}

Nouvelle question de l'utilisateur :
"{user_message}"

Consignes :
- Tiens compte des préférences et des conseils déjà donnés.
- Ne répète pas inutilement les recommandations précédentes.
- Fais référence aux progrès ou aux objectifs de l'utilisateur si c'est pertinent.
- Si de nouvelles informations sont fournies, mets à jour mentalement le profil de l'utilisateur.
- Fournis une réponse claire, cohérente et encourageante.
'''
next_turn_prompt

'\nTu es un coach de santé virtuel.\n\nUtilise le contexte suivant pour personnaliser ta réponse.\n\nContexte :\n{past_context}\n\nNouvelle question de l\'utilisateur :\n"{user_message}"\n\nConsignes :\n- Tiens compte des préférences et des conseils déjà donnés.\n- Ne répète pas inutilement les recommandations précédentes.\n- Fais référence aux progrès ou aux objectifs de l\'utilisateur si c\'est pertinent.\n- Si de nouvelles informations sont fournies, mets à jour mentalement le profil de l\'utilisateur.\n- Fournis une réponse claire, cohérente et encourageante.\n'

In [17]:
import os, subprocess

def run_prompt(prompt: str, model: str | None = None, temperature: float = 0.7, max_new_tokens: int = 120):
    """Send a single-turn prompt via `ollama run`.

    - Set OLLAMA_MODEL env var or pass model to override.
    - Falls back to dry-run if ollama is unavailable."""
    model = model or os.environ.get('OLLAMA_MODEL', 'llama3')
    cmd = ['ollama', 'run', model]
    try:
        proc = subprocess.run(cmd, input=prompt.encode('utf-8'), stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=True)
        out = proc.stdout.decode('utf-8', errors='ignore')
        print(out)
        return out
    except FileNotFoundError:
        print('[dry-run] ollama not installed. Prompt to send:', prompt)
    except subprocess.CalledProcessError as e:
        err = e.stderr.decode('utf-8', errors='ignore') if e.stderr else str(e)
        print('[dry-run] ollama call failed:', err)
    return None
